In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:

        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/anhkhoal0506/tokenised-jazz-music/dataset_test.npy
/kaggle/input/datasets/anhkhoal0506/tokenised-jazz-music/dataset_val.npy
/kaggle/input/datasets/anhkhoal0506/tokenised-jazz-music/A2_seed.mid
/kaggle/input/datasets/anhkhoal0506/tokenised-jazz-music/dataset_train.npy
/kaggle/input/notebooks/anhkhoal0506/music-generation/generated.mid
/kaggle/input/notebooks/anhkhoal0506/music-generation/marjazz.pth
/kaggle/input/notebooks/anhkhoal0506/music-generation/__results__.html
/kaggle/input/notebooks/anhkhoal0506/music-generation/__notebook__.ipynb
/kaggle/input/notebooks/anhkhoal0506/music-generation/__output__.json
/kaggle/input/notebooks/anhkhoal0506/music-generation/generated_stretched.mid
/kaggle/input/notebooks/anhkhoal0506/music-generation/custom.css
/kaggle/input/models/anhkhoal0506/majazz-version7-pth/pytorch/default/1/marjazz.pth
/kaggle/input/pm-116354817-at-05-02-2026-05-51-42/__script__.py
/kaggle/input/pm-116354817-at-05-02-2026-05-51-42/platformdirs-4.9.6-p

In [2]:
from typing import cast

import numpy as np
import torch
import torch.nn as nn
from miditok import REMI, TokSequence, TokenizerConfig
from torch.utils.data import DataLoader, Dataset
import symusic

# ── Paths ────────────────────────────────────────────────────────────────────
TRAIN_PATH = "/kaggle/input/datasets/anhkhoal0506/tokenised-jazz-music/dataset_train.npy"
VAL_PATH   = "/kaggle/input/datasets/anhkhoal0506/tokenised-jazz-music/dataset_val.npy"
TEST_PATH  = "/kaggle/input/datasets/anhkhoal0506/tokenised-jazz-music/dataset_test.npy"
SEED_PATH  = "/kaggle/input/datasets/anhkhoal0506/tokenised-jazz-music/A2_seed.mid"
OUT_PATH   = "/kaggle/working/generated.mid"
MODEL_PATH = "/kaggle/working/marjazz.pth"
# ── Hyperparameters ───────────────────────────────────────────────────────────
EMBED_DIM   = 128
HIDDEN_SIZE = 256
NUM_LAYERS  = 2
MLP_DIM     = 256
DROPOUT     = 0.3
BATCH_SIZE  = 32
MAX_SEQ_LEN = 1024
EPOCHS      = 300
LR          = 1e-3
LR_FACTOR   = 0.5
LR_PATIENCE = 2
ES_PATIENCE = 5
MAX_GEN_LEN = 500

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = REMI(TokenizerConfig(
    pitch_range=(40, 90),
    special_tokens=["PAD", "BOS", "EOS", "MASK"],
    tempo_range=(80, 160),
    use_pitchdrum_tokens=False,
))
vocab_size = len(tokenizer)
print(f"Vocab size: {vocab_size}")

# ── Dataset ───────────────────────────────────────────────────────────────────
class PreTokenizedDataset(Dataset):
    def __init__(self, path: str):
        self.data = torch.from_numpy(np.load(path, allow_pickle=True).astype(np.int64))

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {"input_ids": self.data[idx]}


def make_loaders(
    train_path: str = TRAIN_PATH,
    val_path: str   = VAL_PATH,
    test_path: str  = TEST_PATH,
    batch_size: int = BATCH_SIZE,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    train_ds = PreTokenizedDataset(train_path)
    val_ds   = PreTokenizedDataset(val_path)
    test_ds  = PreTokenizedDataset(test_path)
    print(f"Chunks — train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}")
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size)
    return train_loader, val_loader, test_loader

# ── Model ─────────────────────────────────────────────────────────────────────
class MarJazz(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = EMBED_DIM,
        hidden_size: int = HIDDEN_SIZE,
        num_layers: int = NUM_LAYERS,
        mlp_dim: int = MLP_DIM,
        dropout: float = DROPOUT,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm1 = nn.LSTM(embed_dim, hidden_size, num_layers, batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.drop1 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(hidden_size)

        self.lstm2 = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.drop2 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(hidden_size)

        self.mlp   = nn.Linear(hidden_size, mlp_dim)
        self.act   = nn.GELU()
        self.drop3 = nn.Dropout(dropout)
        self.norm3 = nn.LayerNorm(mlp_dim)

        self.fc = nn.Linear(mlp_dim, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.embedding(x)
        x, _ = self.lstm1(x)
        x = self.norm1(self.drop1(x))
        x, _ = self.lstm2(x)
        x = self.norm2(self.drop2(x))
        x = self.act(self.mlp(x))
        x = self.norm3(self.drop3(x))
        return self.fc(x)

# ── Evaluate ──────────────────────────────────────────────────────────────────
def evaluate(model: nn.Module, dataloader: DataLoader) -> float:
    model.eval()
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    total_loss = 0.0
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            inputs, targets = input_ids[:, :-1], input_ids[:, 1:]
            outputs = model(inputs)
            total_loss += criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1)).item()
    return total_loss / len(dataloader)

# ── Train ─────────────────────────────────────────────────────────────────────
def train(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = LR,
    lr_factor: float = LR_FACTOR,
    lr_patience: int = LR_PATIENCE,
    es_patience: int = ES_PATIENCE,
):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, factor=lr_factor, patience=lr_patience
    )
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    best_val_loss, no_improve = float("inf"), 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            inputs, targets = input_ids[:, :-1], input_ids[:, 1:]
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        train_loss = total_loss / len(train_loader)
        val_loss   = evaluate(model, val_loader)
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch+1}/{epochs}  train={train_loss:.4f}  val={val_loss:.4f}  lr={current_lr:.2e}")
        if val_loss < best_val_loss:
            best_val_loss, no_improve = val_loss, 0
        else:
            no_improve += 1
            if no_improve >= es_patience:
                print(f"Early stopping at epoch {epoch+1} (val loss no improvement for {es_patience} epochs)")
                break

# ── Save / Load model ─────────────────────────────────────────────────────────
def save_model(model: nn.Module, path: str = MODEL_PATH):
    torch.save({
        "model_state": model.state_dict(),
        "hyperparams": {
            "vocab_size":   vocab_size,
            "embed_dim":    EMBED_DIM,
            "hidden_size":  HIDDEN_SIZE,
            "num_layers":   NUM_LAYERS,
            "mlp_dim":      MLP_DIM,
            "dropout":      DROPOUT,
        },
    }, path)
    print(f"Model saved → {path}")


def load_model(path: str = MODEL_PATH) -> nn.Module:
    checkpoint = torch.load(path, weights_only=True, map_location=device)
    model = MarJazz(**checkpoint["hyperparams"])
    model.load_state_dict(checkpoint["model_state"])
    model.to(device)
    model.eval()
    print(f"Model loaded ← {path}")
    return model


Using device: cuda
Vocab size: 184


In [3]:
# model = load_model("/kaggle/input/notebooks/anhkhoal0506/music-generation/marjazz.pth")

In [4]:
# temperature = [0.95, 1.5, 0.8, 0.5, 0.4, 0.75]
# top_k = [10, 20, 30, 40, 50, 60, 70]
# repetition_penalty = [1.5, 1.3, 1.7, 2, 2.5]
# repetition_window = [40, 50, 60, 100, 200, 30]
# num = 0
# lookup = {}
# for temp in temperature:
#     song = generate(model, 
#                     max_length = 2000, 
#                     temperature =  temp,
#                     top_k = 10,
#                     repetition_penalty = 1.5,
#                     repetition_window = 100
#                    )
#     lookup[f"Song${num}"] = {"Temp": temp, "Top_k": 10, "repetition_penalty": 1.5,
#                             "repetition_window": 100}
#     path   = f"/kaggle/working/song_temp_{num}_top_k_10_penalty_1_5_window_100.midi"
#     save_midi(song, path)
#     print("Song ", num, " generated window 100")
#     num += 1
# print(lookup)

In [5]:
# ── Generate ──────────────────────────────────────────────────────────────────
def generate(
    model: nn.Module,
    seed_path: str = SEED_PATH,
    max_length: int = MAX_GEN_LEN,
    temperature: float = 0.95,
    top_k: int = 20,
    repetition_penalty: float = 1.3,
    repetition_window: int = 50
):
    model.eval()
    score = symusic.Score(seed_path)
    generated = cast(TokSequence, tokenizer.encode(score)[0]).ids[:]
 
    with torch.no_grad():
        for _ in range(max_length):
            inputs  = torch.tensor(generated).unsqueeze(0).to(device)
            outputs = model(inputs)
            logits  = outputs[0, -1]  # Raw logits for last position
 
            # 1. Repetition penalty — reduce score of recently used tokens
            recent = generated[-repetition_window:]
            for token_id in set(recent):
                if logits[token_id] > 0:
                    logits[token_id] /= repetition_penalty
                else:
                    logits[token_id] *= repetition_penalty
 
            # 2. Temperature — controls randomness (higher = more creative)
            logits = logits / temperature
 
            # 3. Top-k filtering — only keep the k most likely tokens
            if top_k > 0:
                top_k_values, _ = torch.topk(logits, top_k)
                min_top_k = top_k_values[-1]
                logits[logits < min_top_k] = float('-inf')
 
            # 4. Sample from the distribution (instead of argmax)
            probs = torch.softmax(logits, dim=0)
            next_token = int(torch.multinomial(probs, 1).item())
 
            generated.append(next_token)
 
    return generated

def save_midi(tokens, output_path: str = OUT_PATH):
    score = tokenizer.decode([TokSequence(ids=tokens)])
    score.dump_midi(output_path)
    print(f"MIDI saved → {output_path}")


In [6]:
# ── Run ───────────────────────────────────────────────────────────────────────
train_loader, val_loader, test_loader = make_loaders()

model = MarJazz(vocab_size).to(device)
train(model, train_loader, val_loader)
save_model(model)

test_loss = evaluate(model, test_loader)
print(f"Test loss: {test_loss:.4f}")

tokens = generate(
    model, 
    max_length = 2000, 
    temperature =  0.95,
    top_k = 10,
    repetition_penalty = 1.5,
    repetition_window = 100
)
save_midi(tokens)


Chunks — train: 112462  val: 1119  test: 1141
Epoch 1/300  train=2.6456  val=2.4836  lr=1.00e-03
Epoch 2/300  train=2.4799  val=2.4281  lr=1.00e-03
Epoch 3/300  train=2.4321  val=2.4025  lr=1.00e-03
Epoch 4/300  train=2.4094  val=2.3820  lr=1.00e-03
Epoch 5/300  train=2.3937  val=2.3731  lr=1.00e-03
Epoch 6/300  train=2.3820  val=2.3611  lr=1.00e-03
Epoch 7/300  train=2.3726  val=2.3628  lr=1.00e-03
Epoch 8/300  train=2.3648  val=2.3532  lr=1.00e-03
Epoch 9/300  train=2.3585  val=2.3492  lr=1.00e-03
Epoch 10/300  train=2.3531  val=2.3464  lr=1.00e-03
Epoch 11/300  train=2.3484  val=2.3425  lr=1.00e-03
Epoch 12/300  train=2.3443  val=2.3456  lr=1.00e-03
Epoch 13/300  train=2.3405  val=2.3499  lr=1.00e-03
Epoch 14/300  train=2.3371  val=2.3450  lr=5.00e-04
Epoch 15/300  train=2.3269  val=2.3427  lr=5.00e-04
Epoch 16/300  train=2.3242  val=2.3408  lr=5.00e-04
Epoch 17/300  train=2.3226  val=2.3415  lr=5.00e-04
Epoch 18/300  train=2.3208  val=2.3411  lr=5.00e-04
Epoch 19/300  train=2.3194 

In [7]:

# # ── Stretch ───────────────────────────────────────────────────────────────────
# STRETCHED_PATH = "/kaggle/working/generated_stretched.mid"

# def stretch_midi(tokens, stretch: float = 1.5, output_path: str = STRETCHED_PATH):
#     """
#     Decode tokens and save a version with note durations scaled by stretch.
#     Does not touch note start times — makes notes longer/more legato.
#     Original tokens are untouched; writes to a separate file.
#     """
#     score = tokenizer.decode([TokSequence(ids=tokens)])
#     for track in score.tracks:
#         for note in track.notes:
#             note.duration = max(1, int(note.duration * stretch))
#     score.dump_midi(output_path)
#     print(f"Stretched MIDI (×{stretch}) saved → {output_path}")


# stretch_midi(tokens, stretch=1.5)
